# 4D time-lapse: a CO2 plume

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_timelapse_4d_co2.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 9.3 Reservoir to Seismic
2. 10.4 CO₂ Plume Time-Lapse
3. 9.5 Export Your Model

In [ ]:
# Ask Ogbon starter: Monitor fluid change over time (4D)
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""A 4D (time-lapse) signal: the top of a sand before and after its pore fill changes.

Follows Sections 9.3 (Reservoir to Seismic) and 10.4 (CO2 Plume Time-Lapse):
Gassmann (1951) on the book's soft sand, Wood's law for the mixture, and the
4D signal as the NRMS of Kragh and Christie (2002).
Units: moduli GPa, density g/cc, so the sand's velocities are km/s (the square
root of GPa over g/cc); the shale is in m/s; impedance m/s times g/cc.
Assumptions and limits:
- Gassmann: low frequency, connected pores, the frame unchanged by the
  injection (no pressure or temperature effect on it).
- Wood's law mixes finely (uniform saturation); patchy saturation is stiffer.
- The fills are fixed at the book's anchor, 50 C and 25 MPa (Batzle and Wang 1992).
- One interface under one wavelet: the NRMS of two traces that differ only in
  that coefficient; no tuning, no time shift, no overburden change.
- The field: the change fills the field earth's trap above its contact; baseline
  and monitor carry their own noise, set so that where nothing changed they differ
  by the repeat NRMS; the plume's tuning and the time shifts beneath it are real.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
SCENARIO = 'co2'        # 'co2', 'gas' or 'water'; FLUIDS below were filled in for it
SAT_CHANGE_PCT = 40.0   # share of the pore volume whose fill is replaced (percent)
REPEAT_NRMS_PCT = 20.0  # the repeat noise, the NRMS of two baselines (percent)

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's soft sand at 25 percent porosity (Section 9.3), dry frame from the Hertz-Mindlin soft-sand line
PHI = 0.25                                # porosity (fraction)
K_DRY, G_DRY = 4.183680106235644, 5.103026238493424         # dry frame (GPa)
K_MIN, RHO_MIN = 36.6, 2.65   # quartz, GPa and g/cc
# pore fills at the book's anchor conditions (50 C, 25 MPa): bulk modulus (GPa), density (g/cc)
FLUIDS = {
    'base': (3.0243945863581994, 1.068195),
    'monitor': (0.23393225250780414, 0.8341916637829245)
}
SHALE = (2800, 2.2521031547808517)   # the standard shale: m/s, g/cc
SEAL = (2800, 1241.3793103448277, 2.2521031547808517)   # the standard shale as the field's seal: vp, vs (m/s), g/cc
F_HZ, Q, DT_MS, T_MAX_MS = 30.0, 80.0, 2.0, 1000.0   # the field surveys: Ricker peak (Hz), Q, sample and length (ms)

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def validate(phi, sat_pct, repeat_pct, scenario):
    """Stop with a clear message on a parameter the model cannot use."""
    if scenario not in ('co2', 'gas', 'water'):
        raise ValueError("SCENARIO must be 'co2', 'gas' or 'water', not %r" % (scenario,))
    if not 0 < phi < 1:
        raise ValueError('porosity PHI = %g is out of range: it is a fraction between 0 and 1' % phi)
    if not 0 <= sat_pct <= 100:
        raise ValueError('SAT_CHANGE_PCT = %g is out of range: 0 to 100 percent' % sat_pct)
    if not 0 < repeat_pct < 100 or min(K_DRY, G_DRY, K_MIN, FLUIDS['base'][0], FLUIDS['monitor'][0]) <= 0:
        raise ValueError('the repeat noise must lie between 0 and 100 percent, and every modulus be positive')

def sand(k_fl, den_fl):
    """vp (km/s), vs (km/s) and density (g/cc) of the sand with a pore fill of modulus k_fl (GPa) and density den_fl (g/cc)."""
    # Gassmann (1951): the frame is fixed, the fill changes
    b = 1 - K_DRY / K_MIN
    k_sat = K_DRY + b * b / (PHI / k_fl + (1 - PHI) / K_MIN - K_DRY / K_MIN ** 2)
    den = (1 - PHI) * RHO_MIN + PHI * den_fl
    vp = np.sqrt((k_sat + 4.0 / 3.0 * G_DRY) / den)
    vs = np.sqrt(G_DRY / den)
    return vp, vs, den

def wood_mix(s, base, monitor):
    """Modulus (GPa) and density (g/cc) of a fine mixture, share s of the monitor fill in the base: Wood's law."""
    k_b, den_b = base
    k_m, den_m = monitor
    return 1.0 / ((1 - s) / k_b + s / k_m), (1 - s) * den_b + s * den_m

def coefficient(z, z_shale):
    """Normal-incidence coefficient of the top of the sand under the shale."""
    return (z - z_shale) / (z + z_shale)

def nrms(a, b):
    """The NRMS of two traces or windows, percent (Kragh and Christie 2002)."""
    return 200.0 * np.sqrt(((a - b) ** 2).mean()) / (np.sqrt((a ** 2).mean()) + np.sqrt((b ** 2).mean()))

def survey(pay):
    """The field earth with the trap's sand at pay (km/s, km/s, g/cc) above the contact and the base fill below,
    imaged with the constant-Q wavelet: the clean section, the earth, and its map to time."""
    base = [1e3 * q for q in sand(*FLUIDS['base'])[:2]] + [sand(*FLUIDS['base'])[2]]
    e = field_earth(seal=SEAL, upper=(1e3 * pay[0], 1e3 * pay[1], pay[2]), lower=base)
    t = field_time(e, DT_MS, T_MAX_MS)
    return field_image(reflectivity(t['vp'] * t['den']), DT_MS, F_HZ, q=Q), e, t

def repeat_noise(shape, scale, seed):
    """Noise in the band of the data, scaled to scale (one survey's own; two surveys draw two)."""
    n = field_image(np.random.default_rng(seed).standard_normal(shape), DT_MS, F_HZ, q=Q)
    return n * scale / np.sqrt((n ** 2).mean())

def shift_ms(a, b, k0, k1):
    """The time shift (ms) of b against a in samples k0 to k1: the peak of their correlation, refined by a parabola."""
    lags = np.arange(-8, 9)
    cc = np.array([np.dot(a[k0:k1], np.roll(b, -k)[k0:k1]) for k in lags])
    i = int(np.clip(np.argmax(cc), 1, lags.size - 2))
    return DT_MS * (lags[i] + 0.5 * (cc[i - 1] - cc[i + 1]) / (cc[i - 1] - 2 * cc[i] + cc[i + 1]))

def plot_4d(e, t, b, m, plume_cols, title):
    """Baseline, monitor and their difference over the trap, all at one clip."""
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), sharey=True)
    c = np.percentile(np.abs(b), 99.5)
    xp = e['x'][plume_cols] / 1e3
    tp = t['t'][t['gas'][:, plume_cols].any(axis=1)]
    for a, s, name in zip(ax, (b, m, m - b), ('Baseline', 'Monitor', 'Monitor minus baseline, same clip')):
        show(a, s, e['x'], t['t'], name, clip=c)
        a.set_xlim(xp.min() - 0.4, xp.max() + 0.4)
        a.set_ylim(tp.max() + 200, tp.min() - 120)
        if a is not ax[0]:
            a.set_ylabel('')
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    plt.show()

def main():
    validate(PHI, SAT_CHANGE_PCT, REPEAT_NRMS_PCT, SCENARIO)
    s = SAT_CHANGE_PCT / 100.0
    vp0, vs0, den0 = sand(*FLUIDS['base'])
    vp1, vs1, den1 = sand(*wood_mix(s, FLUIDS['base'], FLUIDS['monitor']))
    z_shale = SHALE[0] * SHALE[1]
    z0, z1 = vp0 * 1000.0 * den0, vp1 * 1000.0 * den1      # km/s to m/s
    r0, r1 = coefficient(z0, z_shale), coefficient(z1, z_shale)
    signal = 200.0 * abs(r0 - r1) / (abs(r0) + abs(r1))   # the 4D signal, Kragh and Christie (2002)
    report('vp_baseline_kms', vp0)
    report('vp_monitor_kms', vp1)
    report('impedance_change_pct', 100.0 * (z1 / z0 - 1), '%.4f')
    report('r_baseline', r0)
    report('r_monitor', r1)
    report('signal_nrms_pct', signal, '%.4f')
    report('repeat_nrms_pct', REPEAT_NRMS_PCT, '%.4f')
    report('detectable', signal > REPEAT_NRMS_PCT, '%d')
    # self-check: an NRMS lies between 0 (identical traces) and 200 percent (opposite ones)
    assert 0.0 <= signal <= 200.0 + 1e-9
    # the field: the change fills the trap above the contact; two surveys whose noise alone gives REPEAT_NRMS_PCT
    b0, e, t = survey((vp0, vs0, den0))
    m0, e1, t1 = survey((vp1, vs1, den1))
    quiet = (t['lay'] < 12).all(axis=1) & (t['t'] > 60)       # the overburden, which nothing changed
    q = REPEAT_NRMS_PCT / 100.0                                # two copies of one window, each with noise of rms
    level = np.sqrt((b0[quiet] ** 2).mean()) * q / np.sqrt(2.0 - q * q)   # this level, differ by an NRMS of q
    b = b0 + repeat_noise(b0.shape, level, 21)
    m = m0 + repeat_noise(m0.shape, level, 22)
    cols = np.nonzero(e['gas'].any(axis=0))[0]                 # the traces over the plume
    rows = (t['t'] > t['t'][t['gas'][:, cols].any(axis=1)].min() - 30) & (t['t'] < t['t'][t1['gas'][:, cols].any(axis=1)].max() + 60)
    ic = cols[np.argmax(e['gas'][:, cols].sum(axis=0))]       # the thickest plume
    pred = 2000.0 * e['dz'] * (1.0 / e1['vp'][:, ic] - 1.0 / e['vp'][:, ic]).sum()
    k0 = int((t1['t'][t1['gas'][:, ic]].max() + 40) / DT_MS)
    clean = nrms(b0[rows][:, cols], m0[rows][:, cols])
    window = nrms(b[rows][:, cols], m[rows][:, cols])
    fieldnote('plume_traces', cols.size, '%d')
    fieldnote('plume_thickest_m', e['gas'][:, ic].sum() * e['dz'], '%.1f')
    fieldnote('repeat_nrms_measured_pct', nrms(b[quiet], m[quiet]), '%.1f')
    fieldnote('plume_window_signal_nrms_pct', clean, '%.1f')
    fieldnote('plume_window_nrms_pct', window, '%.1f')
    fieldnote('plume_clears_repeat', clean > REPEAT_NRMS_PCT, '%d')
    fieldnote('time_shift_predicted_ms', pred, '%.2f')
    fieldnote('time_shift_measured_ms', shift_ms(b0[:, ic], m0[:, ic], k0, k0 + int(250 / DT_MS)), '%.2f')
    plot_4d(e, t, b, m, cols, '4D, %s: %.0f percent of the fill replaced in the trap. NRMS: %.0f percent where nothing changed, '
            '%.0f percent over the plume (%.0f percent without the noise)' % (SCENARIO, SAT_CHANGE_PCT, nrms(b[quiet], m[quiet]), window, clean))

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)